# Chai-1 Structure Prediction — Protein + Mn-Porphyrin Ligands

This notebook predicts the 3-D structure of a target protein bound to three different **Mn-porphyrin ligands** (defined in `smuprot_0.json`, `smuprot_1.json`, and `smuprot_2.json`).

## What Chai-1 does
Chai-1 is a deep-learning structure predictor that takes a sequence-level description of a molecular complex (protein chains + small-molecule ligands) and outputs a predicted 3-D structure.  For ligands it uses the **SMILES string** directly — no docking program or starting pose is needed.  The output is a CIF file containing the predicted coordinates of the entire complex.

## Pipeline overview
```
smuprot_N.json
   │  (json_to_fasta.py)
   ▼
smuprot_N/smuprot_N.fasta        ← protein sequence + ligand SMILES
   │  (run_batch.py → run_prediction.py)
   ▼
smuprot_N/chai1/…/pred.model_idx_*.cif   ← 5 candidate structures
   │  (process_best.py)
   ▼
smuprot_N/<name>.cif / .pdb      ← best structure by confidence score
```

## A note on the Mn atom and the UFF warning
Chai-1 uses RDKit internally to generate an initial 3-D conformer of the ligand before the neural network runs.  RDKit's UFF force-field typer does not have parameters for transition metals like Mn, so it prints a warning:

> `UFFTYPER: Unrecognised atom type: Mn5+2 (12)`

**This is a warning, not an error.** RDKit builds the conformer using its distance-geometry algorithm (ETKDG) first, which does not require force-field atom types and handles metal-containing molecules correctly.  The UFF step is an optional geometry optimisation that fails silently and is skipped.  The **Mn atom and all its bonds are fully preserved** in the conformer that Chai-1 uses, and the SMILES written to the FASTA is identical to the one in the JSON.  The warning is suppressed in `json_to_fasta.py` to avoid confusion during batch runs.

---
## 0. Environment Setup

Mount Google Drive and install dependencies.  
**After the first cell runs `condacolab.install()` the kernel will restart automatically.**  
Re-run all cells from the top after the restart — on the second pass `condacolab` detects it is already installed and skips the restart, so the install cell proceeds directly to `pip install`.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("We are in Google Colab: installing apbs-binary, pymol-open-source, numpy, pandas")
    !pip install -q numpy chai_lab gemmi biopython
else:
    print("We are not in Google Colab: all the packages are asumed to be installed in the current environment")

from importlib.metadata import version

for pkg in ["chai_lab", "gemmi", "biopython"]:
    print(f"{pkg}: {version(pkg)}")


We are not in Google Colab: all the packages are asumed to be installed in the current environment
chai_lab: 0.6.1
gemmi: 0.6.7
biopython: 1.88


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    smuprot_dir = '/content/drive/MyDrive/Samuel/NERSC/BI/R_smuprot/'
else:
    smuprot_dir = './'

import os
os.chdir(smuprot_dir)

---
## 1. JSON → FASTA Conversion

Each `smuprot_N.json` file contains:
- A **protein sequence** (115 amino acids, identical across all three files)
- A **ligand SMILES** (different Mn-porphyrin variant in each file)

The cell below defines a Python function that reads one JSON, extracts these two pieces of information, and writes a Chai-1-format FASTA file.  The FASTA format Chai-1 expects uses tagged headers:

```
>protein|name=<label>
MDELRELLK...
>ligand|name=<label>
c1ccc(cc1)c2c3...[Mn]...
```

The ligand SMILES is copied verbatim from the JSON — the Mn centre is not modified.

In [3]:
import json, os
from pathlib import Path

def json_to_fasta(json_path, verbose=False):
    """Read a smuprot JSON and write a Chai-1 FASTA into a same-named subfolder.

    The Mn atom in the ligand SMILES is kept exactly as-is.
    The RDKit UFF warning about Mn is suppressed (it is non-fatal).
    Returns the path to the written FASTA file.
    """
    # Suppress RDKit's UFF atom-type warning for metal centres
    if not verbose:
        try:
            from rdkit import RDLogger
            RDLogger.DisableLog('rdApp.warning')
            RDLogger.DisableLog('rdApp.error')
        except ImportError:
            pass

    json_path = Path(json_path)
    with open(json_path) as f:
        data = json.load(f)
    entry = data[0] if isinstance(data, list) else data

    name = entry['name']
    protein_seq, ligand_smiles = None, None
    for seq in entry['sequences']:
        if 'proteinChain' in seq:
            protein_seq = seq['proteinChain']['sequence']
        elif 'ligand' in seq:
            ligand_smiles = seq['ligand']['smiles']

    if not protein_seq or not ligand_smiles:
        raise ValueError(f"Could not find protein sequence or ligand in {json_path}")

    # Output folder named after the JSON stem (smuprot_0, smuprot_1, smuprot_2)
    out_dir = json_path.parent / json_path.stem
    out_dir.mkdir(exist_ok=True)
    fasta_path = out_dir / f"{json_path.stem}.fasta"

    fasta_path.write_text(
        f">protein|name={name}_protein\n"
        f"{protein_seq}\n"
        f">ligand|name={name}_ligand\n"
        f"{ligand_smiles}\n"
    )

    print(f"[{json_path.name}]")
    print(f"  Protein : {len(protein_seq)} residues")
    print(f"  Ligand  : {ligand_smiles[:100]}{'...' if len(ligand_smiles)>100 else ''}")
    print(f"  FASTA   : {fasta_path}")
    return fasta_path

---
## 2. Ligand 0 — Mn(III)-Tetraphenylporphyrin with Vinyl Groups

**Source:** `smuprot_0.json`

**Ligand SMILES:** `c1ccc(cc1)c2c3ccc4n3[Mn]56n7c(ccc7c(c8[n+]5c(c4)C=C8)c9ccccc9)cc1[n+]6c2C=C1`

This is a **Mn(III)-porphyrin** in which the porphyrin macrocycle carries four *meso* phenyl groups and two vinyl (`C=C`) substituents on the β-pyrrole positions.  The Mn is coordinated to all four porphyrin nitrogen atoms; the formal charges on two of the pyrrole N atoms (`[n+]`) balance the Mn oxidation state.

Porphyrin-metal complexes are studied as enzyme mimics, photosensitisers, and catalysts.  Mn-porphyrins in particular show SOD (superoxide dismutase) and catalase-like activity and have been investigated as therapeutic antioxidants.

**Expected Chai-1 behaviour:** Chai-1 will predict where on the protein surface this flat, aromatic, charged macrocycle binds.  The planar Mn-porphyrin core (diameter ≈ 10 Å) is a distinctive shape that the model can typically place well.

In [ ]:
# Convert smuprot_0.json → FASTA, then run Chai-1
fasta_0 = json_to_fasta('smuprot_0.json')
print()
!sh run.sh smuprot_0 chai1

[smuprot_0.json]
  Protein : 115 residues
  Ligand  : c1ccc(cc1)c2c3ccc4n3[Mn]56n7c(ccc7c(c8[n+]5c(c4)C=C8)c9ccccc9)cc1[n+]6c2C=C1
  FASTA   : smuprot_0/smuprot_0.fasta
FASTA ready, running Chai-1...

  Chai-1 Prediction Pipeline  —  R_smuprot
Python: Python 3.12.13
Target : smuprot_0
Engine : chai1

----------------------------------------------------------------------
  Processing smuprot_0.json
----------------------------------------------------------------------
[1/2] Converting JSON to FASTA ...

Converting smuprot_0.json ...
  JSON   : smuprot_0.json
  Name   : protein_mn_ligand
  Protein: 115 residues
  Ligand : c1ccc(cc1)c2c3ccc4n3[Mn]56n7c(ccc7c(c8[n+]5c(c4)C=C8)c9ccccc9)cc1[n+]6c2C=C1
  FASTA  : smuprot_0/smuprot_0.fasta
  Done → smuprot_0/smuprot_0.fasta

[2/2] Running Chai-1 (chai1) on folder: smuprot_0 ...
Running predictions for: smuprot_0  [chai1]
Found 1 FASTA file(s)
[1/1] Predicting: smuprot_0
smuprot_0:   0% 0/1 [00:00<?, ?fasta/s]===================================

---
## 3. Ligand 1 — Mn-Porphyrin with Fluorinated Aryl and Hydroxyl/Methoxy Substituents

**Source:** `smuprot_1.json`

**Ligand SMILES:** `FC1=C(...)C(F)=C(F)C(F)=C1F` *(truncated)*

This is a more heavily functionalised Mn-porphyrin.  Key features:

| Substituent | Location | Effect |
|---|---|---|
| Pentafluorophenyl (`C6F5`) | One *meso* position | Strong electron-withdrawing group; increases oxidation potential, improves stability |
| Hydroxyl (`OH`) and methoxy (`OMe`, `OC`) groups | Other *meso* aryl rings | Tunable electron density; potential H-bond donors/acceptors for binding |
| `[Mn+2]` with neutral `[N]` pyrroles | Metal centre | Different formal charge assignment compared to ligand 0, same overall complex |

The pentafluorophenyl group makes this compound more electron-deficient and sterically demanding at one face of the macrocycle, which can direct binding orientation in a protein pocket.  Fluorinated porphyrins are also more resistant to oxidative degradation, making them attractive for in-vivo applications.

**Compared to Ligand 0:** Same protein, same Mn-porphyrin scaffold, but with electron-withdrawing F atoms and H-bonding –OH/–OMe groups that will alter how and where the molecule sits in the binding pocket.

In [4]:
# Convert smuprot_1.json → FASTA, then run Chai-1
fasta_1 = json_to_fasta('smuprot_1.json')
print()
!sh run.sh smuprot_1 chai1

[smuprot_1.json]
  Protein : 115 residues
  Ligand  : FC1=C(C2=C(C=C3)N4C3=CC(C=C5)=[N]6C5=C(C7=CC(C)=C(O)C(C8=NC(C=C(OC)C(OC)=C9)=C9N8)=C7)C%10=CC=C%11N%...
  FASTA   : smuprot_1/smuprot_1.fasta

  Chai-1 Prediction Pipeline  —  R_smuprot


---
## 4. Ligand 2 — Mn-Porphyrin: Alternative Formal-Charge Representation

**Source:** `smuprot_2.json`

**Ligand SMILES:** `FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(...)=[N+]5[Mn-2]...` *(truncated)*

This SMILES describes **the same chemical compound as Ligand 1** but uses a different formal-charge convention:

| Property | Ligand 1 | Ligand 2 |
|---|---|---|
| Mn formal charge | `[Mn+2]` | `[Mn-2]` |
| Pyrrole N charges | two `[N]` (neutral), two `N` | four `[N+]` |
| Net charge of complex | same | same |

In coordination chemistry, the formal charges on a metal complex can be assigned using different bookkeeping conventions (ionic model vs. covalent model).  Both representations are valid SMILES for the same molecule; the electron density around Mn is identical regardless of how the formal charges are partitioned between the metal and the porphyrin N atoms.  RDKit will produce an equivalent 3-D conformer for both.

**Why include both?**  Running Chai-1 on both representations is a useful internal consistency check: the two predictions should give essentially the same binding pose and similar confidence scores.  Significant divergence would indicate a sensitivity of the model to SMILES charge notation, which would be worth reporting.

---
## 5. Ligand 3 — Mn-Porphyrin

**Source:** `smuprot_3.json`

**Ligand SMILES:** `FC1=C(C2=C(C=C3)N4C3=C(C(F)(F)F)C(C=C5)=...`*(truncated)*

This SMILES describes **the same chemical compound as Ligand 1** but uses a different formal-charge convention:

| Property | Ligand 1 | Ligand 2 |
|---|---|---|
| Mn formal charge | `[Mn+2]` | `[Mn-2]` |
| Pyrrole N charges | two `[N]` (neutral), two `N` | four `[N+]` |
| Net charge of complex | same | same |

In [ ]:
# Convert smuprot_3.json → FASTA, then run Chai-1
fasta_3 = json_to_fasta('smuprot_3.json')
print()
!sh run.sh smuprot_3 chai1

[smuprot_3.json]
  Protein : 115 residues
  Ligand  : FC1=C(C2=C(C=C3)N4C3=C(C(F)(F)F)C(C=C5)=[N+]6C5=C(C7=CC(C)=C(O)C(C8=NC(C=C(OC)C(OC)=C9)=C9N8)=C7)C%1...
  FASTA   : smuprot_3/smuprot_3.fasta

  Chai-1 Prediction Pipeline  —  R_smuprot
Python: Python 3.12.13
Target : smuprot_3
Engine : chai1

----------------------------------------------------------------------
  Processing smuprot_3.json
----------------------------------------------------------------------
[1/2] Converting JSON to FASTA ...

Converting smuprot_3.json ...
  JSON   : smuprot_3.json
  Name   : protein_mn_ligand
  Protein: 115 residues
  Ligand : FC1=C(C2=C(C=C3)N4C3=C(C(F)(F)F)C(C=C5)=[N+]6C5=C(C7=CC(C)=C(O)C(C8=NC(C=C(OC)C(OC)=C9)=C9...
  FASTA  : smuprot_3/smuprot_3.fasta
  Done → smuprot_3/smuprot_3.fasta

[2/2] Running Chai-1 (chai1) on folder: smuprot_3 ...
Running predictions for: smuprot_3  [chai1]
Found 1 FASTA file(s)
[1/1] Predicting: smuprot_3
smuprot_3:   0% 0/1 [00:00<?, ?fasta/s]======================

---
## 6a. Ligand 4 — Mn-Porphyrin

**Source:** `smuprot_4.json`

**Ligand SMILES:** `FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)...`*(truncated)*

This SMILES describes **the same chemical compound as Ligand 1** but uses a different formal-charge convention:

| Property | Ligand 1 | Ligand 2 |
|---|---|---|
| Mn formal charge | `[Mn+2]` | `[Mn-2]` |
| Pyrrole N charges | two `[N]` (neutral), two `N` | four `[N+]` |
| Net charge of complex | same | same |

In [ ]:
# Convert smuprot_4.json → FASTA, then run Chai-1
fasta_4 = json_to_fasta('smuprot_4.json')
print()
!sh run.sh smuprot_4 chai1

[smuprot_4.json]
  Protein : 115 residues
  Ligand  : FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)=C5C=CC6=[N+]5[Mn-2]74[N+]8=C2C=CC8=C(C(F)(F)F)...
  FASTA   : smuprot_4/smuprot_4.fasta

  Chai-1 Prediction Pipeline  —  R_smuprot
Python: Python 3.12.13
Target : smuprot_4
Engine : chai1

----------------------------------------------------------------------
  Processing smuprot_4.json
----------------------------------------------------------------------
[1/2] Converting JSON to FASTA ...

Converting smuprot_4.json ...
  JSON   : smuprot_4.json
  Name   : protein_mn_ligand
  Protein: 115 residues
  Ligand : FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)=C5C=CC6=[N+]5[Mn-2]74[N+]8=C2C=CC8=C...
  FASTA  : smuprot_4/smuprot_4.fasta
  Done → smuprot_4/smuprot_4.fasta

[2/2] Running Chai-1 (chai1) on folder: smuprot_4 ...
Running predictions for: smuprot_4  [chai1]
Found 1 FASTA file(s)
[1/1] Predicting: smuprot_4
smuprot_4:   0% 0/1 [00:00<?, ?fasta/s]======================

---
## 6b. Ligand 4 — Mn-Porphyrin protonated

**Source:** `smuprot_4.json`

**Ligand SMILES:** `FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)...`*(truncated)*

This SMILES describes **the same chemical compound as Ligand 1** but uses a different formal-charge convention:

| Property | Ligand 1 | Ligand 2 |
|---|---|---|
| Mn formal charge | `[Mn+2]` | `[Mn-2]` |
| Pyrrole N charges | two `[N]` (neutral), two `N` | four `[N+]` |
| Net charge of complex | same | same |

In [4]:
# Convert smuprot_4_protonated.json → FASTA, then run Chai-1
fasta_4_protonated = json_to_fasta('smuprot_4_protonated.json')
print()
!sh run.sh smuprot_4_protonated chai1

[smuprot_4_protonated.json]
  Protein : 115 residues
  Ligand  : FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)=C5C=CC6=[N+]5[Mn-2]74[N+]8=C2C=CC8=C(C(F)(F)F)...
  FASTA   : smuprot_4_protonated/smuprot_4_protonated.fasta

  Chai-1 Prediction Pipeline  —  R_smuprot
Python: Python 3.12.13
Target : smuprot_4_protonated
Engine : chai1

----------------------------------------------------------------------
  Processing smuprot_4_protonated.json
----------------------------------------------------------------------
[1/2] Converting JSON to FASTA ...

Converting smuprot_4_protonated.json ...
  JSON   : smuprot_4_protonated.json
  Name   : protein_mn_ligand_protonated
  Protein: 115 residues
  Ligand : FC(C(F)=C(F)C(F)=C1F)=C1C2=C3[N+]4=C(C=C3)C(C(F)(F)F)=C5C=CC6=[N+]5[Mn-2]74[N+]8=C2C=CC8=C...
  FASTA  : smuprot_4_protonated/smuprot_4_protonated.fasta
  Done → smuprot_4_protonated/smuprot_4_protonated.fasta

[2/2] Running Chai-1 (chai1) on folder: smuprot_4_protonated ...
Running pred

---
## 5. Results Summary

After the three prediction runs finish, the cell below collects the best-scoring structure from each and prints a comparison table.  

**Interpreting the confidence score:**  
Chai-1 reports an overall confidence metric (similar to pLDDT in AlphaFold) that ranges roughly 0–1.  Higher is better.  Scores above ~0.7 generally indicate a reliable prediction; scores below ~0.5 suggest the model is uncertain (which can happen when the ligand binding site is not well-defined by the sequence alone).

**Output files** (written into each `smuprot_N/` folder):  
- `<name>.cif` — best predicted structure in mmCIF format  
- `<name>.pdb` — same structure in PDB format (converted via biotite or gemmi)  
- `summary_*.txt` — score statistics across all 5 samples

In [ ]:
import json, re
import numpy as np
from pathlib import Path

def collect_results(stem):
    """Read the summary file written by process_best.py for a given smuprot_N folder."""
    folder = Path(stem)
    summaries = sorted(folder.glob('summary_*.txt'))
    if not summaries:
        return None
    txt = summaries[-1].read_text()
    score = None
    pdb   = None
    for line in txt.splitlines():
        if line.startswith('Best score'):
            try: score = float(line.split(':')[1].strip())
            except: pass
        if line.startswith('Output PDB'):
            pdb = line.split(':')[1].strip()
    return {'stem': stem, 'score': score, 'pdb': pdb, 'summary': summaries[-1].name}

print(f"{'JSON':<14} {'Best score':>12} {'PDB file'}")
print('-' * 55)
for stem in ['smuprot_0', 'smuprot_1', 'smuprot_2']:
    r = collect_results(stem)
    if r:
        score_str = f"{r['score']:.4f}" if r['score'] is not None else 'N/A'
        print(f"{stem:<14} {score_str:>12}   {r['pdb'] or 'see folder'}")
    else:
        print(f"{stem:<14} {'(not run yet)':>12}")

JSON             Best score PDB file
-------------------------------------------------------
smuprot_0      (not run yet)
smuprot_1      (not run yet)
smuprot_2      (not run yet)
